In [ ]:
## load packages 
import pandas as pd
import re
import numpy as np
# import plotnine
# from plotnine import *
import pickle

## nltk imports
from nltk.tokenize import word_tokenize, wordpunct_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

## sklearn imports
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## print mult things
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## random
import random

pd.set_option('display.max_colwidth', None)

In [ ]:
# Function that Processes Text
def processtext(one_str, stop_list):
    if pd.isna(one_str):
        return ""
    
    no_stop = [tok for tok in wordpunct_tokenize(one_str)
               if tok not in stop_list]
    
    processed_string = " ".join([
        porter.stem(i.lower())
        for i in no_stop
        if i.lower().isalpha() and len(i) >= 3
    ])
    
    return processed_string

In [ ]:
# Assigning text for commercial and personal binary in vectorization process

def label_post(text):
    if pd.isna(text):
        return 0
    
    text_lower = str(text).lower()
    
    commercial_signals = [
        r'link in bio', r'swipe up', r'use code', r'promo code',
        r'discount', r'affiliate', r'sponsored', r'#ad\\b', r'#spon\\b',
        r'shop now', r'buy now', r'dm me', r'click the link',
        r'\\$\\d+', r'% off', r'free shipping', r'collab'
    ]
    
    for pattern in commercial_signals:
        if re.search(pattern, text_lower):
            return 1
    
    return 0

In [ ]:
# Function that creates a word count table/document term matrix
def create_dtm(list_of_strings, metadata):
  vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
  )
  dtm_sparse = vectorizer.fit_transform(list_of_strings)
  dtm_dense_named = pd.DataFrame(
    dtm_sparse.todense(),
    columns=vectorizer
      .get_feature_names_out()
  )
  dtm_dense_named_withid = pd.concat(
    [metadata.reset_index(),
     dtm_dense_named], axis=1
  )
  return(dtm_dense_named_withid)

In [ ]:
tcm_df = pd.read_csv("../Data/cleanedData.csv")
tcm_df.head()

In [ ]:
porter = PorterStemmer()
list_stopwords = stopwords.words("english")


tcm_df_creator['label'] = tcm_df_creator['text_clean'].apply(label_post)


tcm_df_creator['process_text'] = [processtext(caption, stop_list=list_stopwords) 
                                   for caption in tcm_df_creator['text_clean']]


tcm_creator_dtm = create_dtm(
    tcm_df_creator['process_text'], 
    tcm_df_creator[['label',
                    'process_text', 
                    'text_clean']]
)

In [ ]:
# rebuild clean dataset from the SAME source
tcm_creator_dtm_clean = tcm_creator_dtm.dropna(subset=['label']).copy()

# reset index so rows align
tcm_creator_dtm_clean = tcm_creator_dtm_clean.reset_index(drop=True)

# define y properly (no ravel)
y = tcm_creator_dtm_clean[['label']]

# define X from SAME dataframe
X = tcm_creator_dtm_clean.drop(columns=['label', 'index', 'text_clean', 'process_text'], errors='ignore')

# sanity check (must match)
print(y)
print(X.shape, y.shape)

# split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model
logit_lasso = LogisticRegression(penalty="l1", C=0.01, solver="liblinear", max_iter=100)

# fit
logit_lasso.fit(X_train, y_train)

In [ ]:
y_pred = logit_lasso.predict(X_test)

y_predprob = logit_lasso.predict_proba(X_test)

In [ ]:
y_pred[0:5]
y_predprob[0:5]

In [ ]:
y_pred_df = pd.DataFrame({'y_pred_binary': y_pred,
                         'y_pred_continuous': [one_prob[1] 
                                            for one_prob in y_predprob],
                         'y_true': y_test_man})
y_pred_df.sample(n = 10, random_state = 4484)

In [ ]:
## precision as tp / tp+fp 
error_cond = [(y_pred_df['y_true'] == 1) & (y_pred_df['y_pred_binary'] == 1),
             (y_pred_df['y_true'] == 1) & (y_pred_df['y_pred_binary'] == 0),
              (y_pred_df['y_true'] == 0) & (y_pred_df['y_pred_binary'] == 0)]

error_codeto = ["TP", "FN", "TN"]

y_pred_df['error_cat'] = np.select(error_cond, error_codeto, default = "FP")
y_error = y_pred_df.error_cat.value_counts().reset_index().copy()
y_error.columns = ['cat', 'n']
y_error

### precision
print("Precision is:-----------")
y_error.loc[y_error.cat == "TP", 'n'].iloc[0]/(y_error.loc[y_error.cat == "TP", 'n'].iloc[0] +
                    y_error.loc[y_error.cat == "FP", 'n'].iloc[0])

### recall
print("Recall is:---------------")
y_error.loc[y_error.cat == "TP", 'n'].iloc[0]/(y_error.loc[y_error.cat == "TP", 'n'].iloc[0] +
                    y_error.loc[y_error.cat == "FN", 'n'].iloc[0])